# 🔄 Základní pracovní postupy agentů s Microsoft Foundry (Python)

## 📋 Výukový program orchestrací pracovních postupů

Tento notebook představuje výkonné schopnosti **Workflow Builderu** v rámci Microsoft Agent Framework. Naučte se, jak vytvářet sofistikované, vícekrokové pracovní postupy agentů, které mohou zvládat složité obchodní procesy a bezproblémově koordinovat více AI operací.

> **Poznámka k migraci:** Tento příklad dříve odkazoval na GitHub Models. GitHub Models jsou zastaralé (ukončení v červenci 2026), proto nyní používá **Microsoft Foundry** přes `FoundryChatClient`, který cílí na Azure OpenAI **Responses API**.

## 🎯 Výukové cíle

### 🏗️ **Architektura pracovních postupů**
- **Workflow Builder**: Navrhování a orchestraci složitých vícekrokových procesů
- **Event-Driven Execution (Vykonávání řízené událostmi)**: Zpracování událostí pracovního postupu a stavových přechodů
- **Vizualizace pracovních postupů**: Vytváření a vizualizace struktur pracovních postupů
- **Integrace Microsoft Foundry**: Využití AI modelů v kontextu pracovních postupů

### 🔄 **Orchestrace procesů**
- **Sekvenční operace**: Řetězení více úkolů agenta v logickém pořadí
- **Podmíněná logika**: Implementace rozhodovacích bodů a větvení pracovních postupů
- **Řešení chyb**: Odolné zotavení z chyb a odolnost pracovního postupu
- **Správa stavů**: Sledování a řízení stavu vykonávání pracovního postupu

### 📊 **Vzorové pracovní postupy pro podniky**
- **Automatizace obchodních procesů**: Automatizace složitých organizačních pracovních postupů
- **Koordinace více agentů**: Koordinace více specializovaných agentů
- **Škálovatelné vykonávání**: Navrhování pracovních postupů pro podnikovou úroveň
- **Monitorování a přehlednost**: Sledování výkonnosti pracovních postupů a výsledků

## ⚙️ Požadavky a nastavení

### 📦 **Požadované závislosti**

Nainstalujte Agent Framework s funkcionalitou pracovních postupů:

```bash
pip install agent-framework -U
```

### 🔑 **Konfigurace Microsoft Foundry**

Přihlaste se pomocí Azure CLI (`az login`), aby se mohl `AzureCliCredential` autentizovat, a poté nastavte detaily vašeho projektu Microsoft Foundry.

**Nastavení prostředí (soubor .env):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-5-mini
```

### 🏢 **Případy použití v podnicích**

**Příklady obchodních procesů:**
- **Zákaznické onboardingy**: Vícekrokové ověřování a nastavení pracovních postupů
- **Pipeline obsahu**: Automatizovaná tvorba obsahu, revize a publikace
- **Zpracování dat**: ETL pracovní postupy s AI řízenou transformací
- **Zajišťování kvality**: Automatizované testování a validační procesy

**Výhody pracovních postupů:**
- 🎯 **Spolehlivost**: Deterministické vykonávání s obnovou při chybách
- 📈 **Škálovatelnost**: Řízení automatizace procesů s vysokým objemem
- 🔍 **Přehlednost**: Kompletní auditní stopy a monitorování
- 🔧 **Údržba**: Vizualizace návrhu a modulární komponenty

## 🎨 Vzory návrhu pracovních postupů

### Základní struktura pracovních postupů
```mermaid
graph TD
    A[Start] --> B[Úkol agenta 1]
    B --> C{Bod rozhodnutí}
    C -->|Úspěch| D[Úkol agenta 2]
    C -->|Neúspěch| E[Zpracování chyby]
    D --> F[Konec]
    E --> F
```

**Klíčové komponenty:**
- **WorkflowBuilder**: Hlavní orchestrátor
- **WorkflowEvent**: Zpracování událostí a komunikace
- **WorkflowViz**: Vizualizace pracovního postupu a ladění

Postavme váš první inteligentní pracovní postup! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
# Core components for building sophisticated agent workflows
from agent_framework import WorkflowBuilder, WorkflowEvent, WorkflowViz
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential


In [ ]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

import os                      # 🔧 Environment variable access
from dotenv import load_dotenv # 📁 Secure configuration loading

In [ ]:
# 🔧 Initialize Environment Configuration
# Load Microsoft Foundry project settings from .env file
load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [ ]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [ ]:
reviewer_agent = provider.as_agent(
    name=REVIEWER_NAME,
    instructions=REVIEWER_INSTRUCTIONS,
)

front_desk_agent = provider.as_agent(
    name=FRONTDESK_NAME,
    instructions=FRONTDESK_INSTRUCTIONS,
)


In [ ]:
workflow = (
    WorkflowBuilder(start_executor=front_desk_agent)
    .add_edge(front_desk_agent, reviewer_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")


In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The reviewer (last stage) is the only output here.
events = await workflow.run("I would like to go to Paris.")
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o omezení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o co největší přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Originální dokument v jeho mateřském jazyce by měl být považován za autoritativní zdroj. Pro kritické informace se doporučuje profesionální lidský překlad. Nejsme odpovědní za jakékoli nedorozumění nebo nesprávné interpretace vzniklé použitím tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
